In [ ]:
import fastf1
import pandas as pd

from fastf1 import get_session
from tqdm import tqdm
import numpy as np
import os
import tempfile

# -------------------------
# CACHE (local ephemeral disk — NOT a Unity Catalog Volume)
# -------------------------
# Originally this pointed at a Unity Catalog Volume to survive across
# serverless sessions (local disk here is ephemeral per session). That
# didn't work: FastF1's HTTP cache is a SQLite database, and SQLite needs
# real POSIX file-locking / atomic-rename semantics that an object-storage-
# backed Volume mount doesn't reliably provide — every attempt failed with
# `OperationalError: no such table: responses`, even immediately after a
# clean reset, which is not corruption but a structural incompatibility.
# FastF1 hardcodes SQLite as its cache backend, so there's no config knob
# to point it at something Volume-safe instead.
#
# So: the HTTP cache now lives on local disk, purely as a within-this-
# session optimization (don't re-fetch the same race twice in one run). It
# does NOT survive between sessions, and that's fine — cross-session
# resumability (not re-fetching races already ingested in a prior,
# rate-limit-interrupted run) is now handled at the data level instead: see
# the main ingestion loop below, which checks
# workspace.default.f1_master_lap_dataset for already-ingested races and
# writes each new race to that table incrementally, rather than depending
# on the HTTP cache to persist.
_cache_dir = os.path.join(tempfile.gettempdir(), 'fastf1_cache')
os.makedirs(_cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(_cache_dir)

In [0]:
session = get_session(2023, 'Silverstone', 'R')  # Year, GP name, session type (R=Race, Q=Quali)
session.load()  # downloads data and parses everything

In [0]:
laps = session.laps  # lap-level data (driver, time, stint, tyre, pit stop)
results = session.results  # final positions, fastest lap, team
weather = session.weather_data  # temp, humidity, wind
track_stats = session.track_status

In [0]:
pd.set_option('display.max_columns', None)

In [0]:
laps.loc[laps["TrackStatus"] != "1"].head(1000)

In [0]:
results.head()

In [0]:
weather.head()

In [0]:
track_stats.head(150)

In [0]:
events = fastf1.get_event_schedule(2018)
events.head()

In [0]:
events = fastf1.get_event_schedule(2020)
events = events.loc[events["EventFormat"] == "conventional"]["Location"]
events = list(set(events.tolist()))
print(events)

In [0]:
years = [2018, 2019, 2020, 2021, 2022, 2023]
event_data = {}
for year in years:
    e = fastf1.get_event_schedule(year)
    e = e.loc[e["EventFormat"] != "testing"]["EventName"]
    races = e.tolist()
    event_data[year] = races

In [0]:
print(event_data)
for year, events in event_data.items():
    print(f"\t{year} - {len(events)} - {events}")

In [ ]:
TABLE_NAME = "workspace.default.f1_master_lap_dataset"

# -------------------------
# RESUME SUPPORT
# -------------------------
# Since the HTTP cache no longer persists across sessions (see cell-0),
# cross-session resumability is handled here instead: skip any (Year, Race)
# already present in the target table, so a run interrupted by FastF1's
# 500-calls/hour rate limit can be re-run later and only fetch what's
# still missing.
already_ingested = set()
if spark.catalog.tableExists(TABLE_NAME):
    _existing = spark.table(TABLE_NAME).select("Year", "Race").distinct().toPandas()
    already_ingested = set(zip(_existing["Year"], _existing["Race"]))
    print(f"Resuming: {len(already_ingested)} races already in {TABLE_NAME} — will be skipped.")

master_rows = []

for year in tqdm(event_data, desc="Years"):
    for race in tqdm(event_data[year], desc=f"Processing {year}", leave=False):

        if (year, race) in already_ingested:
            continue

        try:
            # -------------------------
            # LOAD SESSION (RACE)
            # -------------------------
            # telemetry=False: telemetry (car data + position data) is
            # never used anywhere in this notebook (only laps, weather,
            # results, track status), but session.load() fetches it by
            # default. Skipping it cuts the number of underlying requests
            # made per race against FastF1's 500-calls/hour budget.
            session = fastf1.get_session(year, race, 'R')
            session.load(telemetry=False)

            laps = session.laps.copy()
            weather = session.weather_data.copy()
            drivers = session.drivers

            # -------------------------
            # CLEAN WEATHER COLUMNS
            # -------------------------
            weather = weather.rename(columns={
                'AirTemp': 'AirTemp_C',
                'TrackTemp': 'TrackTemp_C',
                'Humidity': 'Humidity_pct',
                'WindSpeed': 'WindSpeed_kmh',
                'Rainfall': 'Rainfall_mm'
            })

            # Sort for merge_asof()
            laps_sorted = laps.sort_values("Time")
            weather_sorted = weather.sort_values("Time")

            # -------------------------
            # MERGE WEATHER INTO LAPS
            # -------------------------
            laps_merged = pd.merge_asof(
                laps_sorted,
                weather_sorted,
                on="Time",
                direction="nearest"
            )

            # -------------------------
            # ENSURE TYRE-RELATED FIELDS EXIST
            # -------------------------
            tyre_fields = ['Compound', 'Stint', 'FreshTyre', 'TyreLife']
            for col in tyre_fields:
                if col not in laps_merged.columns:
                    laps_merged[col] = None

            # -------------------------
            # ADD SESSION METADATA
            # -------------------------
            laps_merged['Year'] = year
            laps_merged['Race'] = race
            laps_merged['Circuit'] = session.event['OfficialEventName']
            laps_merged['Location'] = session.event['Location']
            laps_merged['Country'] = session.event['Country']
            laps_merged['SessionDate'] = session.event['EventDate']

            # -------------------------
            # ADD DRIVER METADATA
            # -------------------------
            driver_info = {}
            for drv in drivers:
                info = session.get_driver(drv)
                driver_info[info['Abbreviation']] = {
                    'DriverNumber': info['DriverNumber'],
                    'BroadcastName': info['BroadcastName'],
                    'TeamColor': info['TeamColor'],
                    'TeamName': info['TeamName']
                }

            laps_merged['DriverNumber'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('DriverNumber')
            )
            laps_merged['TeamName'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('TeamName')
            )
            laps_merged['TeamColor'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('TeamColor')
            )
            laps_merged['BroadcastName'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('BroadcastName')
            )

            # -------------------------
            # TRACK STATUS FLAGS
            # -------------------------
            # 1 = track clear, 2 = yellow, 4 = SC, 5 = VSC, 7 = red flag
            laps_merged['TrackStatus'] = laps_merged['TrackStatus'].fillna("1")

            # -------------------------
            # SECTOR TIMES (important for degradation analysis)
            # -------------------------
            laps_merged['Sector1Time'] = laps_merged['Sector1Time'].fillna(pd.Timedelta(seconds=0))
            laps_merged['Sector2Time'] = laps_merged['Sector2Time'].fillna(pd.Timedelta(seconds=0))
            laps_merged['Sector3Time'] = laps_merged['Sector3Time'].fillna(pd.Timedelta(seconds=0))

            # -------------------------
            # GAP TO CAR AHEAD
            # -------------------------
            # Computed from `Time` (FastF1's cumulative session clock when a
            # lap was completed), not `LapTime` (that lap's own duration).
            # Diffing LapTime only compares how fast two laps were relative
            # to each other; it says nothing about how far apart the cars
            # actually are on track. Diffing Time, sorted by running
            # Position within each lap, gives the real on-track gap.
            laps_merged['Position'] = pd.to_numeric(laps_merged['Position'], errors='coerce')

            _by_pos = laps_merged.sort_values(['LapNumber', 'Position'])
            gap_to_ahead = _by_pos.groupby('LapNumber')['Time'].diff().dt.total_seconds()
            laps_merged['GapToAhead'] = gap_to_ahead.reindex(laps_merged.index)
            laps_merged.loc[laps_merged['Position'].isna(), 'GapToAhead'] = np.nan

            # -------------------------
            # DELTA TO LEADER
            # -------------------------
            # Same fix as above: gap to the race leader (P1) on track,
            # using cumulative Time rather than comparing lap durations.
            leader_time_by_lap = (
                laps_merged.loc[laps_merged['Position'] == 1]
                .drop_duplicates('LapNumber')
                .set_index('LapNumber')['Time']
            )
            laps_merged['DeltaToLeader'] = (
                laps_merged['Time'] - laps_merged['LapNumber'].map(leader_time_by_lap)
            ).dt.total_seconds()
            laps_merged.loc[laps_merged['Position'].isna(), 'DeltaToLeader'] = np.nan

            # -------------------------
            # DELTA TO FIELD AVERAGE
            # -------------------------
            avg_laptimes = laps_merged.groupby("LapNumber")["LapTime"].transform("mean")
            laps_merged["DeltaToAverage"] = (laps_merged["LapTime"] - avg_laptimes)

            # -------------------------
            # PIT STOP PROCESSING (CORRECTED VERSION)
            # -------------------------

            # 1. Ensure PitInTime & PitOutTime are Timedelta
            laps_merged["PitInTime"] = pd.to_timedelta(laps_merged["PitInTime"], errors="coerce").fillna(pd.Timedelta(0))
            laps_merged["PitOutTime"] = pd.to_timedelta(laps_merged["PitOutTime"], errors="coerce").fillna(pd.Timedelta(0))

            # 2. A pit occurs if PitOutTime > PitInTime (FastF1 uses 0 → 0 for non-pit laps)
            laps_merged["HasPit"] = laps_merged["PitOutTime"] > laps_merged["PitInTime"]

            # 3. Compute pit duration ONLY for detected pit laps
            laps_merged["PitDuration"] = np.where(
                laps_merged["HasPit"],
                (laps_merged["PitOutTime"] - laps_merged["PitInTime"]).dt.total_seconds(),
                0
            )

            # Optional: Clean negative/zero durations (bad data in old races)
            laps_merged.loc[laps_merged["PitDuration"] < 0, "PitDuration"] = 0

            # -------------------------
            # RACE LAP NUMBER
            # -------------------------
            laps_merged['RaceLap'] = laps_merged['LapNumber']

            # -------------------------
            # QUALIFYING RESULTS
            # -------------------------
            try:
                # telemetry=False: only quali.results is used below, never
                # laps/telemetry from the qualifying session.
                quali = fastf1.get_session(year, race, 'Q')
                quali.load(telemetry=False)

                quali_results = quali.results[['DriverNumber', 'Position', 'Q1', 'Q2', 'Q3']]
                quali_results = quali_results.rename(columns={
                    'Position': 'QualiPosition',
                    'Q1': 'Q1Time',
                    'Q2': 'Q2Time',
                    'Q3': 'Q3Time'
                })

                laps_merged = laps_merged.merge(
                    quali_results,
                    on='DriverNumber',
                    how='left'
                )

            except Exception as e:
                print(f"⚠️ Qualifying not available for {race} {year}: {e}")
                laps_merged[['QualiPosition', 'Q1Time', 'Q2Time', 'Q3Time']] = None

            # -------------------------
            # FINAL RACE RESULTS
            # -------------------------
            results = session.results[['DriverNumber', 'Position', 'Status', 'Points', 'Time']]
            results = results.rename(columns={
                'Position': 'FinalPosition',
                'Time': 'FinalRaceTime'
            })

            laps_merged = laps_merged.merge(
                results,
                on='DriverNumber',
                how='left'
            )

            # -------------------------
            # NORMALIZE DURATION COLUMNS TO PLAIN SECONDS (PER RACE)
            # -------------------------
            # GapToAhead/DeltaToLeader/PitDuration are already float seconds
            # (computed via .dt.total_seconds() above). Everything else
            # still a raw pandas Timedelta (LapTime, Sector1-3Time,
            # PitInTime, PitOutTime, Q1-3Time) gets converted here too, both
            # for consistency (one unit system across every duration
            # column) and because Spark's Arrow conversion maps pandas
            # Timedelta to DayTimeIntervalType, not the plain numeric type
            # downstream notebooks expect — converting to float seconds
            # before the incremental write below avoids relying on that
            # conversion at all. Done per-race now (not once at the end on
            # a full concatenated frame) because each race is written to
            # the table immediately after this.
            _timedelta_cols = laps_merged.select_dtypes(include=["timedelta64[ns]"]).columns
            for _col in _timedelta_cols:
                laps_merged[_col] = laps_merged[_col].dt.total_seconds()

            # -------------------------
            # WRITE THIS RACE TO THE UNITY CATALOG TABLE IMMEDIATELY
            # -------------------------
            # Incremental (append) rather than batching everything until
            # the end of the loop — so if a later race in this run hits the
            # rate limit and the run dies, every race successfully fetched
            # so far is already durably saved, and the resume-skip logic
            # above will pick up correctly on the next run.
            spark.createDataFrame(laps_merged).write.mode("append").saveAsTable(TABLE_NAME)

            # Race control messages (track limits / penalties, for Plan G)
            rc = session.race_control_messages.copy()
            rc['Year'] = year
            rc['Race'] = race
            rc['Circuit'] = session.event['OfficialEventName']
            _rc_timedelta_cols = rc.select_dtypes(include=["timedelta64[ns]"]).columns
            for _col in _rc_timedelta_cols:
                rc[_col] = rc[_col].dt.total_seconds()
            if not rc.empty:
                spark.createDataFrame(rc).write.mode("append").saveAsTable("workspace.default.f1_race_control_messages")

            # -------------------------
            # APPEND TO MASTER (same-session debug artifact only)
            # -------------------------
            master_rows.append(laps_merged)
            print("-"*100)
            print(f"Success for {year} {race}!!!!!")
            print("-"*100)

        except fastf1.exceptions.RateLimitExceededError:
            # Fail fast for the rest of this run instead of burning through
            # every remaining race with a doomed, instant-fail attempt —
            # the resume-skip logic (already_ingested) means the next run
            # picks up exactly where this one stopped.
            print(f"⏱️ Rate limit hit at {race} {year} — stopping this run early. Re-run later to resume.")
            raise SystemExit("FastF1 rate limit reached — stopping run; re-run later to resume.")

        except Exception as e:
            print(f"⚠️ FAILED FOR {race} {year} → {e}")
            continue


# -------------------------
# SAVE (same-session local artifact only — each race was already written
# to the Unity Catalog table above; this is just for in-session debugging)
# -------------------------
if master_rows:
    master_df = pd.concat(master_rows, ignore_index=True)
    master_df.to_parquet("../data/f1_master_lap_dataset.parquet", index=False)
    print("🎉 LOCAL PARQUET SAVED (session-local only, this run's new races) → ../data/f1_master_lap_dataset.parquet")
    print(f"New rows this run: {len(master_df)}")
else:
    print("No new races fetched this run (all already ingested, or all failed).")

In [ ]:
# -------------------------
# SUMMARY (no write here — each race was already appended to the table
# incrementally inside the loop above)
# -------------------------
# This used to be a final bulk `.write.mode("overwrite")` at the end of the
# run. That's now wrong: with incremental per-race appends (see the loop
# above), an end-of-run overwrite using only `master_rows` (this run's new
# races) would DELETE every race resumed/kept from prior runs that wasn't
# re-fetched this time. So this cell is just a status check.
print(f"Total rows now in {TABLE_NAME}: {spark.table(TABLE_NAME).count()}")